# Programming Assignment 5: Document Classification
**AG News - Naïve Bayes and k-Nearest Neighbor, step by step**

## 📋 Assignment Information

**Course:** Unstructured Data Analysis (2026-2)
**Assignment:** Programming Assignment 5 - Document Classification
**Released:** November 9, 2026 (Mon), 09:00
**Deadline:** November 15, 2026 (Sun), 23:59 (late submissions receive 0 points)
**Total:** 100 points

## 🎯 Objectives

Practice the code of the Week 10 Notebook 1, one small block at a time:

- **Naïve Bayes** (lecture 9-2, chapter 2): a bag-of-words matrix, the lecture's product of likelihoods,
  maximum likelihood estimates with Laplace smoothing, prediction with logarithms, and `MultinomialNB`
- **k-Nearest Neighbor** (lecture 9-2, chapter 3): the Minkowski distance, majority vs weighted voting,
  choosing $k$ on validation data, normalization, and `KNeighborsClassifier`

Everything is inside lecture 9-2 and Notebook 1. The data is a **different** sample of AG News (500 training and
150 test articles per category), so your numbers differ from the class notebook.

## 📝 Instructions

1. `File ▸ Save a copy in Drive`, then rename the notebook to **`Assignment_5_YourName_StudentID.ipynb`** (e.g. `Assignment_5_HongGilDong_2026123456.ipynb`).
2. Fill in every `________` blank. Each blank is short: a name, a number, an operator or a short expression.
   **Do not** change the code outside the blanks and **do not** rename variables: the grader re-runs your notebook.
3. Every TODO ends with a **Check** line: your output should match it.
4. `Runtime ▸ Restart and run all` must finish without an error. **Keep the outputs** in the notebook when you submit.
5. Submit the `.ipynb` file to **"Programming Assignment 5 – Document Classification"** under *Assignments*.

**Fill-in-the-blank only**: 10 TODO items, 24 blanks, 100 points. Each TODO lists its points per blank.

| Part | Content | Points |
|---|---|---|
| 1 | Naïve Bayes : bag of words, the lecture's product, Laplace smoothing, prediction, scikit-learn (TODO 1-5) | 50 |
| 2 | k-NN : Minkowski distance, voting, the best $k$, normalization, scikit-learn (TODO 6-10) | 50 |
| | **Total** | **100** |

## &nbsp;0. Setup (given)
*No TODO in this section.* Run it once and check the printed numbers:
`train : 2000   test : 600` and 500 / 150 articles per category.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
pd.set_option('display.width', 140)
np.set_printoptions(linewidth=140, suppress=True)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week10-document-classification/data/ag_news_assignment.csv"

news = pd.read_csv(DATA_URL)
train = news[news['split'] == 'train'].reset_index(drop=True)
test = news[news['split'] == 'test'].reset_index(drop=True)

CLASSES = ['World', 'Sports', 'Business', 'Sci/Tech']
y_train = train['category'].map(CLASSES.index).to_numpy()      # 0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech
y_test = test['category'].map(CLASSES.index).to_numpy()

print('train :', len(train), '  test :', len(test))
print('train per category :', train['category'].value_counts().to_dict())
print('test per category  :', test['category'].value_counts().to_dict())

## &nbsp;1. Naïve Bayes (50 pts)

### TODO 1: Bag of words (8 points, 4 + 4)

Build the count matrices. The vocabulary is learned from the **training** articles only, and the test articles are
counted with that same vocabulary (lecture p. 9-10, Notebook 1 §1).

**Check:** `X_train : (2000, 5173)   X_test : (600, 5173)`

In [ ]:
# TODO 1 (8 pts): learn the vocabulary on the training set, count both sets

vec = CountVectorizer(stop_words='english', min_df=2)
X_train = vec.________(train['text'])          # learn the vocabulary, then count
X_test = vec.________(test['text'])                # count with the same vocabulary

print('X_train :', X_train.shape, '  X_test :', X_test.shape)

### TODO 2: The lecture's likelihood table (12 points, 4 + 4 + 4)

`P_word` (given) is the table of lecture p. 17. For three **new** reviews, multiply $P(word \mid Pos)$ over the words of the
review, do the same for $Neg$, and pick the class with the larger product (both classes are equally likely, as on p. 18-19).

**Check:**
```
Review A: Pos     3.000e-04   Neg     5.000e-06   ->  Positive
Review B: Pos     1.200e-08   Neg     3.150e-06   ->  Negative
Review C: Pos     8.000e-06   Neg     3.000e-06   ->  Positive
```

In [ ]:
# TODO 2 (12 pts): product of the likelihoods, and the decision

P_word = pd.DataFrame({                    # the table of lecture p. 17 (given)
    'Pos': [0.1, 0.1, 0.1, 0.4, 0.2, 0.3, 0.5, 0.1, 0.02, 0.1, 0.02, 0.02, 0.15],
    'Neg': [0.1, 0.1, 0.1, 0.01, 0.2, 0.05, 0.05, 0.1, 0.3, 0.1, 0.35, 0.02, 0.15],
}, index=['this', 'movie', 'was', 'awesome', 'i', 'really', 'enjoyed', 'it',
          'boring', 'and', 'waste', 'of', 'time'])

reviews = {'Review A': 'i really enjoyed this movie',
           'Review B': 'boring movie and waste of time',
           'Review C': 'this awesome movie was boring'}

for name, text in reviews.items():
    words = text.________()                              # the list of words
    pos = P_word.loc[words, 'Pos'].________()             # product of P(word | Pos)
    neg = P_word.loc[words, 'Neg'].prod()               # product of P(word | Neg)
    label = 'Positive' if pos ________ neg else 'Negative'
    print(f'{name}: Pos {pos:13.3e}   Neg {neg:13.3e}   ->  {label}')

### TODO 3: Maximum likelihood estimates with Laplace smoothing (12 points, 4 + 4 + 4)

`counts` (given) holds $count(w, c)$: one row per class, one column per word. Complete the prior and the smoothed
likelihood (lecture p. 16 and p. 20):

$$\hat P(c_j) = \frac{N.Doc(C = c_j)}{\text{Total number of documents}} \qquad
\hat P(w_i \mid c_j) = \frac{count(w_i, c_j) + 1}{\sum_{w \in V} count(w, c_j) + |V|}$$

**Check:** `prior : [0.25 0.25 0.25 0.25]   |V| = 5173   each row sums to : [1. 1. 1. 1.]   smallest P(w|c) : 6.18e-05`

In [ ]:
# TODO 3 (12 pts): prior, vocabulary size, add-1 smoothing

counts = np.vstack([np.asarray(X_train[y_train == c].sum(axis=0)).ravel()     # count(w, c)  (given)
                    for c in range(len(CLASSES))])

prior = np.array([(y_train == c).________() for c in range(len(CLASSES))])      # share of documents in class c
V = X_train.shape[________]                                                          # vocabulary size |V|
P_laplace = (counts + ________) / (counts.sum(axis=1, keepdims=True) + V)            # Laplace (add-1) smoothing

print('prior :', prior, '  |V| =', V, '  each row sums to :', P_laplace.sum(axis=1).round(6),
      f'  smallest P(w|c) : {P_laplace.min():.2e}')

### TODO 4: Predict with logarithms (10 points, 5 + 5)

A product of many small probabilities is computed as a **sum of logarithms**: the class with the largest
$\log \hat P(c) + \sum_i count(w_i) \log \hat P(w_i \mid c)$ is the class with the largest product. For all test articles
at once, the sum is one matrix product. Pick, for every test article, the class with the largest score.

**Check:** `log_scores : (600, 4)   accuracy (by hand) : 0.865`

In [ ]:
# TODO 4 (10 pts): log scores and the predicted class

log_scores = X_test @ np.________(P_laplace).T + np.log(prior)      # one score per article and class
pred_hand = log_scores.________(axis=1)                           # the class with the largest score

print('log_scores :', log_scores.shape, '  accuracy (by hand) :', accuracy_score(y_test, pred_hand))

### TODO 5: The same classifier in scikit-learn (8 points, 4 + 4)

Train `MultinomialNB` with Laplace smoothing on the count matrix and compare with TODO 4.

**Check:** `accuracy (scikit-learn) : 0.865   same predictions as by hand : True`

In [ ]:
# TODO 5 (8 pts): MultinomialNB with add-1 smoothing

nb = MultinomialNB(________=1.0)                 # 1.0 = Laplace smoothing
nb.________(X_train, y_train)                      # estimate P(c) and P(w | c)
pred_nb = nb.predict(X_test)

print('accuracy (scikit-learn) :', accuracy_score(y_test, pred_nb),
      '  same predictions as by hand :', (pred_nb == pred_hand).all())

## &nbsp;2. k-Nearest Neighbor (50 pts)

### TODO 6: Minkowski distance (8 points, 4 + 4)

$$distance(P, Q) = \Big(\sum_{i=1}^{n} \lvert x_i - y_i \rvert^{p}\Big)^{1/p}$$

(lecture p. 27). The table `ref` (given) is the reference data of p. 26; compute the distance of a new point
$x = (1.0, 2.0)$ to every row, for $p = 1$ and $p = 2$.

**Check:**
```
       Vision  Finance  Class  p = 1  p = 2
Doc 1    2.54     0.15  TPAMI   3.39  2.407
Doc 2    1.78     0.00  TPAMI   2.78  2.147
Doc 3    0.12     2.65    JoF   1.53  1.094
Doc 4    0.25     3.52    JoF   2.27  1.695
Doc N    3.12     0.14  TPAMI   3.98  2.820
```

In [ ]:
# TODO 6 (8 pts): Minkowski distance with order p

def minkowski(P, Q, p):
    return (np.abs(P - Q) ** ________).sum() ** (________)


ref = pd.DataFrame({'Vision': [2.54, 1.78, 0.12, 0.25, 3.12], 'Finance': [0.15, 0.00, 2.65, 3.52, 0.14],
                    'Class': ['TPAMI', 'TPAMI', 'JoF', 'JoF', 'TPAMI']}, index=['Doc 1', 'Doc 2', 'Doc 3', 'Doc 4', 'Doc N'])
x_new = np.array([1.0, 2.0])
for p in [1, 2]:
    ref[f'p = {p}'] = [minkowski(x_new, r, p) for r in ref[['Vision', 'Finance']].to_numpy()]
print(ref.round(3))

### TODO 7: Majority voting vs weighted voting (12 points, 4 + 4 + 4)

Five neighbors of a new document, with their classes and distances (given). Majority voting: the share of neighbors in a
class. Weighted voting: each neighbor has weight $1 / distance$, and the weights are normalized to sum to 1 (lecture p. 30).

**Check:** `weights : [0.453 0.226 0.151 0.113 0.057]   majority P(X in TPAMI) = 0.6   weighted P(X in TPAMI) = 0.396`

In [ ]:
# TODO 7 (12 pts): weights and the two voting rules

lab = np.array(['JoF', 'TPAMI', 'JoF', 'TPAMI', 'TPAMI'])      # classes of N1 ... N5  (given)
d = np.array([0.5, 1.0, 1.5, 2.0, 4.0])                         # their distances       (given)

w = 1 / ________                                       # weight = 1 / distance
w = w / w.________()                                 # normalize: the weights sum to 1
p_majority = np.________(lab == 'TPAMI')            # share of TPAMI neighbors
p_weighted = w[lab == 'TPAMI'].sum()              # weight of TPAMI neighbors

print('weights :', w.round(3), '  majority P(X in TPAMI) =', p_majority,
      '  weighted P(X in TPAMI) =', round(p_weighted, 3))

### TODO 8: The best $k$ on validation data (14 points, 4 + 5 + 5)

The TF-IDF matrices and a validation split of the training set are given. Try every odd $k$ from 1 to 41, measure the
error on the **validation** data, and keep the $k$ with the smallest validation error (lecture p. 28 and p. 31).

**Check:** `validation error, k = 1, 3, 5 : [0.228, 0.196, 0.158]   best k : 9`

In [ ]:
# given: TF-IDF matrices and a validation split of the training set
tfidf = TfidfVectorizer(stop_words='english', min_df=2)
T_train = tfidf.fit_transform(train['text'])
T_test = tfidf.transform(test['text'])
T_tr, T_val, y_tr, y_val = train_test_split(T_train, y_train, test_size=0.25, random_state=7, stratify=y_train)
print('T_tr :', T_tr.shape, '  T_val :', T_val.shape)

In [ ]:
# TODO 8 (14 pts): validation error for every candidate k, and the best k

ks = list(range(1, 42, 2))
val_err = []
for k in ks:
    knn = KNeighborsClassifier(________=k)
    knn.fit(T_tr, y_tr)
    val_err.append(1 - knn.score(________, y_val))          # error on the validation data

best_k = ks[int(np.________(val_err))]                     # the k with the smallest validation error

print('validation error, k = 1, 3, 5 :', [round(e, 3) for e in val_err[:3]], '  best k :', best_k)

### TODO 9: Normalization (8 points, 4 + 4)

*Normalization or scaling must be done before finding k-nearest neighbors* (lecture p. 32). Standardize every column of
the table (given): subtract the column average and divide by the column standard deviation.

**Check:**
```
   Height  Weight    BFS
1   1.477   1.616 -0.532
2  -0.434  -0.708  0.798
3   0.347   0.232 -0.665
4  -1.216  -0.874  1.330
N  -0.174  -0.266 -0.931
```

In [ ]:
# TODO 9 (8 pts): z-score of every column

people = pd.DataFrame({'Height': [187, 165, 174, 156, 168], 'Weight': [93, 51, 68, 48, 59],
                       'BFS': [15, 25, 14, 29, 12]}, index=['1', '2', '3', '4', 'N'])      # (given)

z = (people - people.________()) / people.________()          # column average, column standard deviation
print(z.round(3))

### TODO 10: k-NN in scikit-learn with weighted voting (8 points, 4 + 4)

Train `KNeighborsClassifier` on the whole training set with the best $k$ of TODO 8 and **weighted** voting
($1 / distance$), and compute its accuracy on the test set.

**Check:** `k = 9   test accuracy : 0.838`

In [ ]:
# TODO 10 (8 pts): weighted k-NN and its test accuracy

knn = KNeighborsClassifier(n_neighbors=best_k, weights=________)     # weighted voting
knn.fit(T_train, y_train)
acc_knn = knn.________(T_test, y_test)                                    # accuracy on the test set

print('k =', best_k, '  test accuracy :', round(acc_knn, 3))

---
### ✅ Before you submit

- [ ] `Runtime ▸ Restart and run all` finishes with no error
- [ ] Every `________` is filled in and every output cell is visible
- [ ] Every **Check** line of the assignment matches your output
- [ ] The file is named `Assignment_5_YourName_StudentID.ipynb`